In [ ]:
from sklearn.neighbors import KernelDensity
from scipy.signal import find_peaks

def segment_hands_using_kde(binary_mask, center_point, debug=False):
    """
    Splits a binary mask of clock hands into individual lines/masks using KDE.
    
    Args:
        binary_mask (np.array): 2D array (H, W) with 0 and 1.
        center_point (tuple): (x, y) coordinates of the clock center.
    """
    h, w = binary_mask.shape
    y_indices, x_indices = np.nonzero(binary_mask)
    
    if len(x_indices) == 0:
        return []

    # 1. Convert pixels to vectors relative to center
    vectors_x = x_indices - center_point[0]
    vectors_y = y_indices - center_point[1]
    
    # 2. Calculate angles (radians)
    angles = np.arctan2(vectors_y, vectors_x)
    
    # 3. Fit KDE to find dense angular regions (the hands)
    # We pad angles to handle the -pi/pi wrapping
    angles_reshaped = angles.reshape(-1, 1)
    kde = KernelDensity(kernel='gaussian', bandwidth=0.05).fit(angles_reshaped)
    
    # Evaluate KDE on a grid
    grid = np.linspace(-np.pi, np.pi, 1000)[:, None]
    log_dens = kde.score_samples(grid)
    density = np.exp(log_dens)
    
    # 4. Find Peaks (representing the direction of hands)
    peaks, _ = find_peaks(density, height=np.mean(density))
    
    hand_angles = grid[peaks].flatten()
    
    # 5. Assign pixels to nearest peak (Simple clustering based on angle)
    hand_masks = []
    for peak_angle in hand_angles:
        # Create a mask for this specific hand
        # We can select pixels within +/- threshold of the peak angle
        threshold = 0.2 # radians
        
        # Handle wrapping logic for difference calculation if needed
        diff = np.abs(angles - peak_angle)
        diff = np.minimum(diff, 2*np.pi - diff)
        
        hand_pixels_mask = (diff < threshold)
        
        single_hand_mask = np.zeros_like(binary_mask)
        single_hand_mask[y_indices[hand_pixels_mask], x_indices[hand_pixels_mask]] = 1
        hand_masks.append(single_hand_mask)
        
    return hand_masks, hand_angles

# --- Example Usage ---
# Assuming 'pred_mask' is output from Task 3 and 'center' from Task 2
# pred_mask = (model_seg(img) > 0.5).squeeze().cpu().numpy()
# center = (112, 112) # Example
# hands, angles = segment_hands_using_kde(pred_mask, center)

# Visualization
import matplotlib.pyplot as plt
# plt.imshow(pred_mask, cmap='gray')
# for angle in angles:
#     # Draw line from center with angle
#     pass